**Nome:** *JOÃO PAULO DO ESPÍRITO SANTO QUEIROZ*  
**Matrícula:** *4052026000865*  
**Data:** *26/09/2026*  
**Dataset:** *CDC Diabetes Health Indicators (BRFSS 2015)*  
**Fonte original:** https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators  
**Arquivo de origem:** diabetes_012_health_indicators_BRFSS2015.csv  
**Ambiente utilizado:** Databricks Free Edition  
**Tecnologia utilizada:** PySpark 

# MVP — Engenharia de Dados
## Pipeline de Dados BRFSS 2015 — Diabetes

Neste trabalho estou desenvolvendo um pipeline de Engenharia de Dados utilizando PySpark no ambiente Databricks, a partir da base de dados **BRFSS 2015 (Behavioral Risk Factor Surveillance System)**.

Essa base contém informações relacionadas à saúde, hábitos e características dos participantes e também foi utilizada no meu projeto anterior de Machine Learning sobre diabetes.

Para organizar o processamento dos dados, estou utilizando a **Arquitetura Medalhão**, dividida em três camadas:

- **Bronze:** onde ficam armazenados os dados da forma como foram carregados originalmente;
- **Silver:** onde os dados passam pelo processo de tratamento e preparação;
- **Gold:** onde os dados já tratados são organizados para facilitar as análises e a apresentação dos resultados.

## Contexto e perguntas do trabalho

Neste trabalho utilizo a base BRFSS 2015, que reúne informações relacionadas à saúde, hábitos e características dos participantes. Como o conjunto de dados possui informações sobre diabetes e outros indicadores de saúde, o objetivo é organizar esses dados por meio de um pipeline para permitir sua análise.

A partir dos dados disponíveis, pretendo responder às seguintes perguntas:

1. Como os participantes estão distribuídos entre as classificações sem diabetes, pré-diabetes e diabetes?
2. Qual é o perfil do BMI dos participantes da base?
3. Quais indicadores de saúde e hábitos apresentam diferenças entre os grupos relacionados ao diabetes?

Essas perguntas irão orientar a preparação dos dados e as análises realizadas ao longo do trabalho.

## 1. Camada Bronze — Ingestão e validação inicial

Nesta primeira etapa faço a leitura da tabela **`bronze_brfss_diabetes_2015`**, criada a partir do arquivo CSV original.

Em seguida, verifico a quantidade de registros e de colunas da base e visualizo alguns registros para conferir se os dados foram carregados corretamente.

A base possui **253.680 registros e 22 colunas**, confirmando que os dados necessários para continuar o desenvolvimento do pipeline foram carregados.


In [0]:
# MVP - Engenharia de Dados
# Pipeline BRFSS 2015 - Diabetes
# Arquitetura Medalhão: Bronze -> Silver -> Gold

bronze = spark.table("workspace.default.bronze_brfss_diabetes_2015")

print("Quantidade de registros:", bronze.count())
print("Quantidade de colunas:", len(bronze.columns))

display(bronze.limit(10))

Quantidade de registros: 253680
Quantidade de colunas: 22


Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
0.0,1.0,1.0,1.0,25.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,2.0,0.0,1.0,10.0,6.0,8.0
0.0,1.0,0.0,1.0,30.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,0.0,14.0,0.0,0.0,9.0,6.0,7.0
0.0,1.0,1.0,1.0,25.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,3.0,0.0,0.0,1.0,0.0,11.0,4.0,4.0
2.0,1.0,1.0,1.0,30.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,5.0,30.0,30.0,1.0,0.0,9.0,5.0,1.0
0.0,0.0,0.0,1.0,24.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,1.0,8.0,4.0,3.0


### Verificação da estrutura dos dados

Depois de carregar a tabela da camada Bronze, verifico a estrutura dos dados para identificar o nome das colunas, os tipos de dados e se elas permitem valores nulos.

Essa verificação é importante para entender como os dados foram carregados antes de começar o processo de tratamento e preparação para a camada Silver.

In [0]:
# Verificação da estrutura da camada Bronze

bronze.printSchema()

root
 |-- Diabetes_012: double (nullable = true)
 |-- HighBP: double (nullable = true)
 |-- HighChol: double (nullable = true)
 |-- CholCheck: double (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: double (nullable = true)
 |-- Stroke: double (nullable = true)
 |-- HeartDiseaseorAttack: double (nullable = true)
 |-- PhysActivity: double (nullable = true)
 |-- Fruits: double (nullable = true)
 |-- Veggies: double (nullable = true)
 |-- HvyAlcoholConsump: double (nullable = true)
 |-- AnyHealthcare: double (nullable = true)
 |-- NoDocbcCost: double (nullable = true)
 |-- GenHlth: double (nullable = true)
 |-- MentHlth: double (nullable = true)
 |-- PhysHlth: double (nullable = true)
 |-- DiffWalk: double (nullable = true)
 |-- Sex: double (nullable = true)
 |-- Age: double (nullable = true)
 |-- Education: double (nullable = true)
 |-- Income: double (nullable = true)



### Catálogo de Dados

A base BRFSS 2015 utilizada neste trabalho possui 22 colunas, sendo a variável `Diabetes_012` utilizada para representar a classificação relacionada ao diabetes. As demais variáveis apresentam informações de saúde, hábitos e características dos participantes.

A seguir, apresento o catálogo das variáveis utilizadas no desenvolvimento do pipeline.

| Variável | Descrição |
|---|---|
| Diabetes_012 | Classificação de diabetes: 0 = sem diabetes, 1 = pré-diabetes, 2 = diabetes |
| HighBP | Indicador de pressão arterial alta |
| HighChol | Indicador de colesterol alto |
| CholCheck | Realização de exame de colesterol |
| BMI | Índice de Massa Corporal |
| Smoker | Indicador de histórico de tabagismo |
| Stroke | Indicador de ocorrência de AVC |
| HeartDiseaseorAttack | Indicador de doença cardíaca ou infarto |
| PhysActivity | Prática de atividade física |
| Fruits | Consumo de frutas |
| Veggies | Consumo de vegetais |
| HvyAlcoholConsump | Indicador de consumo elevado de álcool |
| AnyHealthcare | Possui algum tipo de cobertura/assistência de saúde |
| NoDocbcCost | Deixou de consultar médico devido ao custo |
| GenHlth | Avaliação do estado geral de saúde |
| MentHlth | Número de dias com saúde mental não considerada boa |
| PhysHlth | Número de dias com saúde física não considerada boa |
| DiffWalk | Dificuldade para caminhar ou subir escadas |
| Sex | Sexo do participante |
| Age | Faixa etária |
| Education | Nível de escolaridade |
| Income | Faixa de renda |


### Verificação de valores nulos

Depois de verificar a estrutura da base, faço uma análise para identificar se existem valores nulos nas colunas.

Essa etapa é importante porque a presença de dados ausentes pode interferir nas próximas etapas de tratamento e também nas análises que serão realizadas. Com essa verificação, consigo saber se será necessário realizar algum tratamento antes de seguir para a camada Silver.

In [0]:
from pyspark.sql.functions import col, sum

nulos = bronze.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in bronze.columns
])

display(nulos)

Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


#### Resultado da verificação

A verificação mostrou que não existem valores nulos nas colunas da base de dados. Dessa forma, não foi necessário realizar tratamento para dados ausentes antes de seguir para as próximas etapas do pipeline.

### Verificação de registros duplicados

Além dos valores nulos, também verifico se existem registros duplicados na base de dados.

Essa análise permite identificar quantos registros aparecem de forma repetida e qual o percentual que eles representam no conjunto de dados. Neste momento, faço apenas a identificação dessas duplicidades, sem remover nenhum registro, para preservar os dados originais e avaliar posteriormente como eles devem ser tratados.

In [0]:
# Verificação de registros duplicados

total = bronze.count()
unicos = bronze.dropDuplicates().count()
duplicados = total - unicos

print("Total de registros:", total)
print("Registros únicos:", unicos)
print("Registros duplicados:", duplicados)
print("Percentual de duplicados:", round((duplicados / total) * 100, 2), "%")

Total de registros: 253680
Registros únicos: 229781
Registros duplicados: 23899
Percentual de duplicados: 9.42 %



## 2. Camada Silver — Tratamento e preparação dos dados

Depois de analisar os dados da camada Bronze, começo a etapa de tratamento e preparação da base para a camada Silver.

Nesta etapa, faço a conversão dos tipos de dados para deixá-los mais adequados às informações que representam. Como a maioria das variáveis possui valores inteiros, essas colunas são convertidas para o tipo inteiro. A coluna **BMI** é mantida como tipo decimal, pois representa o índice de massa corporal.

Após essas transformações, os dados tratados são armazenados em uma nova tabela da camada Silver, mantendo a base Bronze sem alterações.

In [0]:
from pyspark.sql.functions import col

# Criação da camada Silver
silver = bronze

# Converte para inteiro todas as colunas, exceto BMI
for coluna in silver.columns:
    if coluna != "BMI":
        silver = silver.withColumn(coluna, col(coluna).cast("int"))

# Mantém BMI como double
silver = silver.withColumn("BMI", col("BMI").cast("double"))

# Salva a camada Silver
silver.write.mode("overwrite").saveAsTable(
    "workspace.default.silver_brfss_diabetes_2015"
)

print("Tabela Silver criada com sucesso!")
print("Registros:", silver.count())
print("Colunas:", len(silver.columns))

Tabela Silver criada com sucesso!
Registros: 253680
Colunas: 22



### Validação da camada Silver

Depois de criar a camada Silver, faço algumas verificações para confirmar se os dados foram preparados corretamente e também para conhecer melhor algumas informações importantes da base.

Primeiro, verifico a distribuição da variável **Diabetes_012**, que é a variável que indica a situação relacionada ao diabetes. Em seguida, analiso algumas estatísticas do **BMI**, como média, valores mínimo e máximo e os principais percentis.

Com essas verificações, consigo ter uma visão inicial da distribuição dos dados antes de seguir para a próxima etapa do pipeline.


In [0]:
# Validação da camada Silver

silver = spark.table("workspace.default.silver_brfss_diabetes_2015")

print("Distribuição da variável alvo Diabetes_012:")
silver.groupBy("Diabetes_012").count().orderBy("Diabetes_012").show()

print("Estatísticas do BMI:")
silver.select("BMI").summary().show()

Distribuição da variável alvo Diabetes_012:
+------------+------+
|Diabetes_012| count|
+------------+------+
|           0|213703|
|           1|  4631|
|           2| 35346|
+------------+------+

Estatísticas do BMI:
+-------+------------------+
|summary|               BMI|
+-------+------------------+
|  count|            253680|
|   mean|28.382363607694735|
| stddev|6.6086942014060055|
|    min|              12.0|
|    25%|              24.0|
|    50%|              27.0|
|    75%|              31.0|
|    max|              98.0|
+-------+------------------+




### Resultados da validação

A validação mostra que a variável **Diabetes_012** apresenta uma diferença considerável na quantidade de registros entre as três classes. A classe 0 possui 213.703 registros, a classe 1 possui 4.631 e a classe 2 possui 35.346. Isso mostra que a base possui um desbalanceamento entre as classes, principalmente em relação à classe 1.

Na análise do **BMI**, foi encontrada uma média de aproximadamente 28,38 e mediana de 27. Os valores variam entre 12 e 98, sendo que 50% dos registros possuem BMI de até 27.

Esses resultados ajudam a conhecer melhor o comportamento da base após o tratamento e servem como referência para as próximas etapas do pipeline.

%md

## Análise das perguntas do trabalho

### Pergunta 2 — Como o BMI se comporta entre as classificações de diabetes?

Para responder a esta pergunta, comparo o BMI médio dos participantes entre as três classificações da variável Diabetes_012.

In [0]:
from pyspark.sql.functions import avg, round

# BMI médio por classificação de diabetes

analise_bmi = (
    silver
    .groupBy("Diabetes_012")
    .agg(
        round(avg("BMI"), 2).alias("BMI_medio")
    )
    .orderBy("Diabetes_012")
)

display(analise_bmi)

Diabetes_012,BMI_medio
0,27.74
1,30.72
2,31.94



#### Resultado

Os resultados mostram um aumento do BMI médio entre as classificações analisadas. O grupo sem diabetes apresentou BMI médio de 27,74, enquanto o grupo com pré-diabetes apresentou 30,72 e o grupo com diabetes 31,94.

Dessa forma, nesta base de dados, os grupos classificados com pré-diabetes e diabetes apresentam BMI médio maior que o grupo sem diabetes.



### Pergunta 3 — Quais indicadores de saúde e hábitos apresentam diferenças entre os grupos relacionados ao diabetes?

Para esta análise, comparo alguns indicadores presentes na base: pressão alta, colesterol alto, atividade física e tabagismo.

In [0]:
from pyspark.sql.functions import avg, round

analise_indicadores = (
    silver
    .groupBy("Diabetes_012")
    .agg(
        round(avg("HighBP") * 100, 2).alias("HighBP_percentual"),
        round(avg("HighChol") * 100, 2).alias("HighChol_percentual"),
        round(avg("PhysActivity") * 100, 2).alias("PhysActivity_percentual"),
        round(avg("Smoker") * 100, 2).alias("Smoker_percentual")
    )
    .orderBy("Diabetes_012")
)

display(analise_indicadores)

Diabetes_012,HighBP_percentual,HighChol_percentual,PhysActivity_percentual,Smoker_percentual
0,37.11,37.92,77.91,42.97
1,62.9,62.08,67.85,49.28
2,75.27,67.01,63.05,51.82



#### Resultado

Os resultados mostram diferenças entre os grupos analisados. A presença de pressão alta aumenta de 37,11% no grupo sem diabetes para 62,90% no grupo com pré-diabetes e 75,27% no grupo com diabetes.

O colesterol alto também apresenta percentuais maiores nos grupos com pré-diabetes e diabetes, passando de 37,92% no grupo sem diabetes para 62,08% e 67,01%, respectivamente.

Em relação aos hábitos, a prática de atividade física diminui entre os grupos, passando de 77,91% no grupo sem diabetes para 67,85% no pré-diabetes e 63,05% no diabetes. Já o percentual de participantes com histórico de tabagismo aumenta de 42,97% para 49,28% e 51,82%.

Esses resultados mostram que, nesta base, os grupos relacionados ao pré-diabetes e diabetes apresentam diferenças importantes nos indicadores de saúde e hábitos analisados.


## 3. Camada Gold – Preparação dos dados para análise

Após realizar o tratamento e a validação dos dados na camada Silver, sigo para a construção da camada Gold.

Nesta etapa, os dados já tratados serão organizados e agregados para facilitar a análise das informações relacionadas ao diabetes. A partir da camada Gold, será possível obter uma visão mais clara da distribuição dos dados e gerar informações que poderão ser utilizadas nas análises finais do projeto.

In [0]:
from pyspark.sql.functions import col, round

# Criação da camada Gold
# Agrupa os dados pela classificação de diabetes
total_registros = silver.count()

gold = (
    silver
    .groupBy("Diabetes_012")
    .count()
    .withColumnRenamed("count", "quantidade")
    .withColumn(
        "percentual",
        round((col("quantidade") / total_registros) * 100, 2)
    )
    .orderBy("Diabetes_012")
)

display(gold)

Diabetes_012,quantidade,percentual
0,213703,84.24
1,4631,1.83
2,35346,13.93


In [0]:
# Salva a camada Gold como tabela
gold.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_brfss_diabetes_2015")

print("Tabela Gold criada com sucesso!")
print("Registros:", gold.count())

Tabela Gold criada com sucesso!
Registros: 3


In [0]:
# Validação da camada Gold

gold_final = spark.table("workspace.default.gold_brFSS_diabetes_2015")

print("Dados armazenados na camada Gold:")
display(gold_final)

Dados armazenados na camada Gold:


Diabetes_012,quantidade,percentual
0,213703,84.24
1,4631,1.83
2,35346,13.93



### Pergunta 1 — Como os participantes estão distribuídos entre as classificações de diabetes?

A análise da camada Gold mostra que 84,24% dos participantes estão classificados como sem diabetes, 1,83% como pré-diabetes e 13,93% como diabetes.

A maior parte dos registros está concentrada na classe sem diabetes, mostrando que existe um desbalanceamento entre as três classificações da variável Diabetes_012.


## Conclusão

Neste trabalho desenvolvi um pipeline de Engenharia de Dados utilizando PySpark no Databricks, seguindo a Arquitetura Medalhão com as camadas Bronze, Silver e Gold.

Na camada Bronze trabalhei com os dados originais da base BRFSS 2015, verificando sua estrutura, valores nulos e registros duplicados.

Na camada Silver realizei o tratamento e a padronização dos tipos dos dados, mantendo os registros da base e preparando as informações para as análises.

Por fim, na camada Gold organizei os dados de forma agregada, apresentando a quantidade e o percentual de registros para cada classificação da variável Diabetes_012. Dessa forma, é possível visualizar com mais clareza a distribuição das classes e o desbalanceamento existente na base de dados.

Com isso, o pipeline permite acompanhar o processo desde os dados originais até uma estrutura mais organizada e preparada para análise.


## Autoavaliação

Considero que consegui atingir os principais objetivos propostos para este MVP. Foi possível realizar a ingestão dos dados no Databricks, organizar o pipeline seguindo as camadas Bronze, Silver e Gold, verificar a qualidade dos dados e realizar as análises definidas no início do trabalho.

Durante o desenvolvimento, também consegui aplicar na prática conceitos de Engenharia de Dados e relacionar este trabalho com o projeto de Machine Learning desenvolvido anteriormente utilizando a mesma base.

Como ponto de melhoria, considero que o pipeline pode ser ampliado futuramente com novas transformações, análises e formas de automatização do processamento dos dados.